## Búsqueda de parámetros

Para el último paso, vamos a buscar los mejores parámetros para nuestro modelo.

Ya sabmeos que aumentar el número de samples no ofrecía ganancias que valiesen la pena, por lo que usaremos el subconjunto de 15000 ejemplos de cada clase que hemos estado utilizando hasta ahora.

Realizamos una búsqueda de parámetros aleatoria seguido de una búsqueda de grid sobre un rango creado de los mejores parámetros encontrados por la búsqueda aleatoria.
* Los parámetros de este proceso de tuning están detallados en el archivo **src/experiment_config/tuning_15k.yaml**.
* El código está en **src/tuning_pipeline.py**.


Desafortunadamente, incluso tras haber realizado esta búsqueda, no hemos conseguido ninguna mejora sustancial.

Los mejores parámetros que ha encontrado son:
* "colsample_bytree": 0.6
* "learning_rate": 0.1
* "max_depth": 7
* "min_child_weight": 5
* "n_estimators": 500
* "subsample": 1

Sin embargo, la puntuación de f1_macro es: 0.7241. Es muy similar al resultado durante la comparación (0.7192).

El cuello de botella de este rendimiento probablemente se deba al *overlap* que hemos visto entre las clases. Por ejemplo, "DoS" y "DDoS" siguen confundiéndose y las clases de "Benign", "Recon", "Spoofing" y "Web-based" son inerentemente similares.

LLegar más lejos requeriría probar a crear nuevas features más discriminativas (*feature engineering*). 

Otro camino puede ser utilizar redes neuronales que quizás sean capaces de capturar relaciones no-lineales que XGBoost no ha podido encontrar.

## Red Neuronal (MLP)

Como alternativa a XGBoost, probamos una red neuronal MLP básica para comprobar si es capaz de capturar relaciones no-lineales que los modelos tradicionales no han podido encontrar.

* El código está en **src/nn_pipeline.py** y los parámetros del experimento en **src/experiment_config/mlp_15k.yaml**.
* La arquitectura utilizada es: `input → 256 → 128 → 64 → output`, con BatchNorm, Dropout y Early Stopping.

Los resultados obtenidos muestran que el MLP alcanza un accuracy de 0.70 y un f1_macro de 0.70, frente al 0.72 de XGBoost en ambas métricas. El MLP no consigue superar a XGBoost en este dataset tabular, lo cual es un resultado habitual — los modelos basados en árboles suelen ser más robustos con datos tabulares.

XGBoost con los parámetros encontrados en la búsqueda sigue siendo el mejor modelo para este problema, con un f1_macro de 0.7241.

## Evaluación final en el conjunto de test

Como paso final, entrenamos XGBoost con los mejores parámetros encontrados sobre el conjunto de entrenamiento completo (15.000 instancias por clase) y evaluamos sobre un subconjunto balanceado del conjunto de test.

Los resultados obtenidos son consistentes con los experimentos anteriores: accuracy de 0.7534 y f1_macro de 0.7052, una caída pequeña y esperada respecto al 0.7241 obtenido durante la búsqueda de parámetros. No hay signos de *overfitting* — el modelo generaliza bien a datos no vistos.

<img src="../results/plots/tuning_15k_test_confusion_matrix.png" width="600">

La matriz de confusión es prácticamente idéntica a la obtenida durante el entrenamiento, lo que confirma que los patrones observados son estructurales y no artefactos del conjunto de entrenamiento:
* "Mirai" sigue siendo perfectamente clasificado (1.00).
* La confusión entre "DDoS" y "DoS" se mantiene estable (~0.30), reflejando la similitud inherente entre ambos tipos de ataque.
* "BruteForce", "Recon" y "Web-based" siguen siendo las clases más difíciles, con tendencia a confundirse entre ellas y con tráfico "Benign".

**Conclusión:** el modelo XGBoost con parámetros optimizados alcanza un rendimiento de 0.70 f1_macro de manera consistente tanto en entrenamiento como en test. El techo de rendimiento está determinado por el *overlap* inherente entre clases, y no por la cantidad de datos ni por la complejidad del modelo. Estás conclusiones son respaldadas por los experimentos con la curva de aprendizaje, búsqueda de parámetros y comparación con una red neuronal MLP.